<a href="https://colab.research.google.com/github/shaymaAziz/Arabic-Domain-Semantic-Space-Lab/blob/main/Shayma_Azhrani_GATE_Arabic_Domain_Semantic_Space_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Arabic Semantic Space Lab — GATE

**Objective:**
Create eight semantic examples within a single domain, then test whether
`Omartificial-Intelligence-Space/GATE-AraBert-v1` places the positive sentence closer to the anchor sentence than the negative sentences.

كل مثال يحتوي على:

- **Anchor:** الجملة الأصلية.
- **Positive:** نفس المعنى بصياغة مختلفة.
- **Hard negative:** كلمات/موضوع قريب، لكن المعنى مختلف في نقطة مهمة.
- **Easy negative:** معنى واضح الاختلاف.

في النهاية ستحصل على خريطة تفاعلية، مصفوفة تشابه، أقرب الجمل، وتحميل بياناتك بصيغتي JSONL وCSV.

> لا تستخدم أسماء حقيقية، أرقام هويات/حسابات، بيانات صحية حقيقية، أسرار عمل، أو نصوصاً منسوخة.

## قواعد جودة البيانات

1. اكتب الجمل بنفسك وبالعربية الطبيعية.
2. حافظ الـPositive على **كل** المعنى، وليس الموضوع فقط.
3. اجعل الـHard negative قريباً لفظياً أو موضوعياً، مع تغيير دلالي مهم: نفي، رقم، زمن، كيان، شرط، سماح/منع، أو حالة مكتملة/معلّقة.
4. اجعل الـEasy negative مختلفاً بوضوح.
5. لا تكرر الجمل بين الأمثلة.
6. استخدم آلية Hard negative متنوعة عبر الأمثلة الثمانية.

In [ ]:
%pip -q install -U sentence-transformers umap-learn plotly pandas scikit-learn

In [ ]:
import json, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

MODEL_ID = "Omartificial-Intelligence-Space/GATE-AraBert-v1"
MODEL_REVISION = "e8537da79d1870992cd573094ef194cf2d151a73"
N_TRIPLETS = 8
ROLES = ["anchor", "positive", "hard_negative", "easy_negative"]
ROLE_AR = {
    "anchor": "الأصلية", "positive": "الموجبة",
    "hard_negative": "السالبة الصعبة", "easy_negative": "السالبة السهلة"
}
print("جاهز ✅")

جاهز ✅


## 1) بيانات المشارك والمجال

استخدم رمز المشارك الذي أعطاك إياه المدرب، واختر **مجالاً واحداً** لكل الأمثلة.

In [ ]:
PARTICIPANT_ID = "P110"       # مثال: P012
DOMAIN = "history"          # المجال المخصص لك بالإنجليزية
DOMAIN_AR = "حقائق تاريخية "   # اسم المجال بالعربية

assert re.fullmatch(r"P\d{3}", PARTICIPANT_ID), "استخدم رمزاً مثل P012"
assert DOMAIN.strip() and DOMAIN_AR.strip(), "أدخل المجال بالعربية والإنجليزية"
print(f"المشارك: {PARTICIPANT_ID} | المجال: {DOMAIN_AR}")

المشارك: P110 | المجال: حقائق تاريخية 


## 2) مثال توضيحي — لا يُضاف إلى بياناتك

لاحظ أن السالب الصعب يستخدم كلمات قريبة لكنه يقلب المعنى بواسطة **تجنب**.

In [ ]:
DEMO = {
    "anchor": "يجب على المريض تناول الدواء بعد الطعام",
    "positive": "ينبغي للمريض أخذ العلاج عقب تناول الوجبة",
    "hard_negative": "يجب على المريض تجنب تناول الدواء بعد الطعام",
    "easy_negative": "تأخرت رحلة الطيران بسبب الأحوال الجوية",
    "hard_negative_mechanism": "negation_or_reversal",
    "difficulty": "hard",
    "naturalness": 5,
    "notes": "تجنّب تقلب الإجراء المطلوب"
}
pd.DataFrame([DEMO]).T.rename(columns={0: "المثال"})

,المثال
anchor,يجب على المريض تناول الدواء بعد الطعام
positive,ينبغي للمريض أخذ العلاج عقب تناول الوجبة
hard_negative,يجب على المريض تجنب تناول الدواء بعد الطعام
easy_negative,تأخرت رحلة الطيران بسبب الأحوال الجوية
hard_negative_mechanism,negation_or_reversal
difficulty,hard
naturalness,5
notes,تجنّب تقلب الإجراء المطلوب


## 3) اكتب أمثلتك الثمانية

الآليات المسموحة للسالب الصعب:
`negation`, `number`, `time`, `entity`, `condition`, `reversed_relation`, `permission_prohibition`, `status_intent`, `other`.

استبدل النصوص الفارغة. يجب أن تكون `naturalness` من 1 إلى 5.

In [ ]:
# ============================================================
# PARTICIPANT DATA ENTRY
# Create 8 semantic triplets within your assigned domain.
# Replace every "..." with your own Arabic sentence or answer.
# ============================================================

TRIPLETS = [
# Triplet 1
{
    "anchor": "ما هي عاصمة الدولة العباسية في أوج عصرها الذهبي؟",
    "positive": "بغداد هي التي كانت عاصمة العباسيين ومقر حكمهم في العصر الذهبي.",
    "hard_negative": "سامراء هي المدينة التي اتخذها بعض خلفاء بني العباس عاصمة لفترة وجيزة.",
    "easy_negative": "القاهرة كانت عاصمة الدولة الفاطمية في مصر.",
    "hard_negative_mechanism": "changed_location",
    "difficulty": "easy",
    "naturalness": 5,
    "notes": "نموذج أساسي يعتمد على الحقائق التاريخية المباشرة مع التمييز بين المكان الأصلي والمؤقت."
},

# Triplet 2
{
    "anchor": "ما هي المعركة التاريخية التي أنهت الوجود الإسلامي في الأندلس عام 1492م؟",
    "positive": "سقوط غرناطة وتسليمها للملوك الكاثوليك هو الحدث الذي أنهى الوجود الإسلامي في الأندلس عام 1492م.",
    "hard_negative": "معركة بلاط الشهداء كانت معركة حاسمة توقف بعدها التمدد الإسلامي في عمق القارة الأوروبية.",
    "easy_negative": "معركة حطين كانت معركة فاصلة بين المسلمين والكرادلة/الصليبيين في الشرق الأدنى.",
    "hard_negative_mechanism": "changed_time",
    "difficulty": "easy",
    "naturalness": 5,
    "notes": "تم الاعتماد على الأحداث والمحطات التاريخية الكبرى مع تغيير الحقبة الزمنية للمعارك."
},

# Triplet 3
{
    "anchor": "ما هي الأسباب المباشرة التي أدت إلى اندلاع الحرب العالمية الأولى عام 1914؟",
    "positive": "اغتيال الأرشيدوق فرنسيس فرديناند ولي عهد النمسا في سراييفو كان الشرارة المباشرة التي أشعلت الحرب العالمية الأولى.",
    "hard_negative": "التنافس الاستعماري ونظام التحالفات السرية والتسابق العسكري بين الدول الأوروبية الكبرى في القرن التسع عشر.",
    "easy_negative": "غزو ألمانيا للبولندا عام 1939 كان السبب المباشر لاندلاع الحرب العالمية الثانية.",
    "hard_negative_mechanism": "changed_condition",
    "difficulty": "medium",
    "naturalness": 5,
    "notes": "الهدف قياس قدرة النموذج على التمييز بين السبب المباشر والأسباب السياقية."
},

# Triplet 4
{
    "anchor": "ما هي الإصلاحات الاقتصادية والسياسية التي تبناها ميخائيل غورباتشوف وأدت لتفكك الاتحاد السوفيتي؟",
    "positive": "سياساتا البيريسترويكا (إعادة الهيكلة الاقتصادية) والجلاسنوست (الشفافية والانفتاح السياسي).",
    "hard_negative": "سياسة خطة السنوات الخمس والتصنيع السريع التي فرضها جوزيف ستالين لبناء الاقتصاد السوفيتي.",
    "easy_negative": "سياسة الباب المفتوح والإصلاحات الاقتصادية التي أطلقها دينج شياو بينج في الصين.",
    "hard_negative_mechanism": "changed_entity",
    "difficulty": "medium",
    "naturalness": 5,
    "notes": "يركز على التمييز بين المصطلحات السياسية وحكام نفس الكيان."
},

# Triplet 5
{
    "anchor": "ما هي المعاهدة التي أنهت حرب السنوات السبع عام 1763 وأعادت رسم الخارطة الاستعمارية في أمريكا الشمالية؟",
    "positive": "معاهدة باريس لعام 1763 التي تنازلت بموجبها فرنسا عن معظم أراضيها في أمريكا الشمالية لبريطانيا.",
    "hard_negative": "معاهدة باريس لعام 1783 التي اعترفت بريطانيا بموجبها باستقلال الولايات المتحدة الأمريكية.",
    "easy_negative": "معاهدة فرساي لعام 1919 التي وضعت شروط السلام بعد نهاية الحرب العالمية الأولى.",
    "hard_negative_mechanism": "changed_time",
    "difficulty": "medium",
    "naturalness": 5,
    "notes": "اختبار قوي جداً للنظر في الفروق الزمنية مع تشابه أطراف واسم المعاهدة."
},

# Triplet 6
{
    "anchor": "كيف ساهمت أزمة الكساد الكبير عام 1929 في صعود التيار النازي إلى السلطة في ألمانيا؟",
    "positive": "أدت الأزمة إلى انهيار الاقتصاد الألماني وارتفاع البطالة، مما أضعف جمهورية فايمار وزاد من شعبية الخطاب الراديكالي للحزب النازي ووعوده بالإنعاش الاقتصاد.",
    "hard_negative": "فرض عقوبات وشروط قاسية على ألمانيا بموجب معاهدة فرساي عام 1919 مما أثار الشعور القومي والرغبة في الانتقام لدى الشعب الألماني.",
    "easy_negative": "تأميم المصانع والشركات الكبرى في روسيا بعد نجاح الثورة البلشفية عام 1917.",
    "hard_negative_mechanism": "changed_condition",
    "difficulty": "hard",
    "naturalness": 5,
    "notes": "يعتمد على التمييز بين الأسباب المتعددة لظاهرة تاريخية بناءً على العامل المحدد في السؤال."
},

# Triplet 7
{
    "anchor": "ما هي الفروق الرئيسية بين نظام الإقطاع في أوروبا والجمهورية الرومانية من حيث الهيكل الطبقي؟",
    "positive": "الإقطاع اعتمد على رابطة الولاء بين السادة والفرسان والأقنان حول الأرض، بينما تكونت الجمهورية الرومانية من طبقتي الأشراف والعامة مع وجود مؤسسات سياسية كالسناتو.",
    "hard_negative": "تكون المجتمع في الإمبراطورية الرومانية المتأخرة من طبقة النبلاء والفرسان والعبيد مع الاعتماد الكامل على الاقتصاد الزراعي المملوك للطبقة الحاكمة.",
    "easy_negative": "نظام الطوائف الاجتماعي في الهند القديمة والذي يقسم المجتمع بناءً على العقيدة والولادة.",
    "hard_negative_mechanism": "changed_intent",
    "difficulty": "hard",
    "naturalness": 5,
    "notes": "يقيس قدرة النموذج على المقارنة التحليلية بين المناهج والأنظمة عبر العصور."
},

# Triplet 8
{
    "anchor": "ما هو الدور الذي لعبته حركة التنوير في التمهيد للثورة الفرنسية، بخلاف الأزمات المالية والمجاعية؟",
    "positive": "نشر أفكار الفلاسفة مثل روسو ومونتيسكيو التي نقدت الحق الإلهي للملوك وطالبت بالعقد الاجتماعي وفصل السلطات، مما قوض شرعية النظام القديم فكرياً.",
    "hard_negative": "تراكم الديون على الخزانة الفرنسية بسبب المشاركة في حرب الاستقلال الأمريكية وسوء المحاصيل الزراعية مما أدى لارتفاع أسعار الخبز وانتفاضة عامة الشعب.",
    "easy_negative": "اختراع المطبعة على يد يوهان جوتنبرج في القرن الخامس عشر مما ساهم في نشر الكتب والعلوم في أوروبا.",
    "hard_negative_mechanism": "negation",
    "difficulty": "hard",
    "naturalness": 5,
    "notes": "نموذج يختبر التعامل مع قيود الاستبعاد (Constraints Handling) في الاستعلام."
}
]

# Allowed values for hard_negative_mechanism:
# negation, changed_number, changed_time, changed_location,
# changed_entity, changed_condition, reversed_relationship,
# changed_intent, permission_prohibition, completed_pending, other

# Allowed values for difficulty:
# easy, medium, hard

# naturalness must be an integer from 1 to 5.

# Display the entered data as a table
import pandas as pd

triplets_df = pd.DataFrame(TRIPLETS)
triplets_df

,anchor,positive,hard_negative,easy_negative,hard_negative_mechanism,difficulty,naturalness,notes
0,ما هي عاصمة الدولة العباسية في أوج عصرها الذهبي؟,بغداد هي التي كانت عاصمة العباسيين ومقر حكمهم في العصر الذهبي.,سامراء هي المدينة التي اتخذها بعض خلفاء بني العباس عاصمة لفترة وجيزة.,القاهرة كانت عاصمة الدولة الفاطمية في مصر.,changed_location,easy,5,نموذج أساسي يعتمد على الحقائق التاريخية المباشرة مع التمييز بين المكان الأصلي والمؤقت.
1,ما هي المعركة التاريخية التي أنهت الوجود الإسلامي في الأندلس عام 1492م؟,سقوط غرناطة وتسليمها للملوك الكاثوليك هو الحدث الذي أنهى الوجود الإسلامي في الأندلس عام 1492م.,معركة بلاط الشهداء كانت معركة حاسمة توقف بعدها التمدد الإسلامي في عمق القارة الأوروبية.,معركة حطين كانت معركة فاصلة بين المسلمين والكرادلة/الصليبيين في الشرق الأدنى.,changed_time,easy,5,تم الاعتماد على الأحداث والمحطات التاريخية الكبرى مع تغيير الحقبة الزمنية للمعارك.
2,ما هي الأسباب المباشرة التي أدت إلى اندلاع الحرب العالمية الأولى عام 1914؟,اغتيال الأرشيدوق فرنسيس فرديناند ولي عهد النمسا في سراييفو كان الشرارة المباشرة التي أشعلت الحرب العالمية الأولى.,التنافس الاستعماري ونظام التحالفات السرية والتسابق العسكري بين الدول الأوروبية الكبرى في القرن التسع عشر.,غزو ألمانيا للبولندا عام 1939 كان السبب المباشر لاندلاع الحرب العالمية الثانية.,changed_condition,medium,5,الهدف قياس قدرة النموذج على التمييز بين السبب المباشر والأسباب السياقية.
3,ما هي الإصلاحات الاقتصادية والسياسية التي تبناها ميخائيل غورباتشوف وأدت لتفكك الاتحاد السوفيتي؟,سياساتا البيريسترويكا (إعادة الهيكلة الاقتصادية) والجلاسنوست (الشفافية والانفتاح السياسي).,سياسة خطة السنوات الخمس والتصنيع السريع التي فرضها جوزيف ستالين لبناء الاقتصاد السوفيتي.,سياسة الباب المفتوح والإصلاحات الاقتصادية التي أطلقها دينج شياو بينج في الصين.,changed_entity,medium,5,يركز على التمييز بين المصطلحات السياسية وحكام نفس الكيان.
4,ما هي المعاهدة التي أنهت حرب السنوات السبع عام 1763 وأعادت رسم الخارطة الاستعمارية في أمريكا الشمالية؟,معاهدة باريس لعام 1763 التي تنازلت بموجبها فرنسا عن معظم أراضيها في أمريكا الشمالية لبريطانيا.,معاهدة باريس لعام 1783 التي اعترفت بريطانيا بموجبها باستقلال الولايات المتحدة الأمريكية.,معاهدة فرساي لعام 1919 التي وضعت شروط السلام بعد نهاية الحرب العالمية الأولى.,changed_time,medium,5,اختبار قوي جداً للنظر في الفروق الزمنية مع تشابه أطراف واسم المعاهدة.
5,كيف ساهمت أزمة الكساد الكبير عام 1929 في صعود التيار النازي إلى السلطة في ألمانيا؟,أدت الأزمة إلى انهيار الاقتصاد الألماني وارتفاع البطالة، مما أضعف جمهورية فايمار وزاد من شعبية الخطاب الراديكالي للح...,فرض عقوبات وشروط قاسية على ألمانيا بموجب معاهدة فرساي عام 1919 مما أثار الشعور القومي والرغبة في الانتقام لدى الشعب ...,تأميم المصانع والشركات الكبرى في روسيا بعد نجاح الثورة البلشفية عام 1917.,changed_condition,hard,5,يعتمد على التمييز بين الأسباب المتعددة لظاهرة تاريخية بناءً على العامل المحدد في السؤال.
6,ما هي الفروق الرئيسية بين نظام الإقطاع في أوروبا والجمهورية الرومانية من حيث الهيكل الطبقي؟,الإقطاع اعتمد على رابطة الولاء بين السادة والفرسان والأقنان حول الأرض، بينما تكونت الجمهورية الرومانية من طبقتي الأش...,تكون المجتمع في الإمبراطورية الرومانية المتأخرة من طبقة النبلاء والفرسان والعبيد مع الاعتماد الكامل على الاقتصاد الز...,نظام الطوائف الاجتماعي في الهند القديمة والذي يقسم المجتمع بناءً على العقيدة والولادة.,changed_intent,hard,5,يقيس قدرة النموذج على المقارنة التحليلية بين المناهج والأنظمة عبر العصور.
7,ما هو الدور الذي لعبته حركة التنوير في التمهيد للثورة الفرنسية، بخلاف الأزمات المالية والمجاعية؟,نشر أفكار الفلاسفة مثل روسو ومونتيسكيو التي نقدت الحق الإلهي للملوك وطالبت بالعقد الاجتماعي وفصل السلطات، مما قوض شر...,تراكم الديون على الخزانة الفرنسية بسبب المشاركة في حرب الاستقلال الأمريكية وسوء المحاصيل الزراعية مما أدى لارتفاع أس...,اختراع المطبعة على يد يوهان جوتنبرج في القرن الخامس عشر مما ساهم في نشر الكتب والعلوم في أوروبا.,negation,hard,5,نموذج يختبر التعامل مع قيود الاستبعاد (Constraints Handling) في الاستعلام.


## 4) التحقق من الجودة

لن يبدأ التحليل حتى تكتمل الأمثلة الثمانية وتنجح قواعد التحقق.

In [ ]:
ALLOWED_MECHANISMS = {
    "negation",
    "changed_number",
    "changed_time",
    "changed_location",
    "changed_entity",
    "changed_condition",
    "reversed_relationship",
    "changed_intent",
    "permission_prohibition",
    "completed_pending",
    "other"
}
ALLOWED_DIFFICULTY = {"easy", "medium", "hard"}
ARABIC_RE = re.compile(r"[\u0600-\u06FF]")

def validate_triplets(rows):
    errors, seen = [], {}
    if len(rows) != N_TRIPLETS:
        errors.append(f"يجب إدخال {N_TRIPLETS} أمثلة بالضبط")
    for i, row in enumerate(rows, 1):
        for field in ROLES:
            value = row.get(field)
            if not isinstance(value, str) or not value.strip():
                errors.append(f"المثال {i}: الحقل {field} فارغ")
                continue
            text = " ".join(value.split())
            if len(text) < 12:
                errors.append(f"المثال {i}: {field} قصير جداً")
            if not ARABIC_RE.search(text):
                errors.append(f"المثال {i}: {field} لا يحتوي نصاً عربياً")
            key = re.sub(r"\s+", " ", text).strip()
            if key in seen:
                errors.append(f"الجملة مكررة في المثالين {seen[key]} و{i}")
            else:
                seen[key] = i
        if row.get("anchor", "").strip() == row.get("positive", "").strip():
            errors.append(f"المثال {i}: الـPositive نسخة مطابقة للـAnchor")
        if row.get("hard_negative_mechanism") not in ALLOWED_MECHANISMS:
            errors.append(f"المثال {i}: آلية السالب الصعب غير صحيحة")
        if row.get("difficulty") not in ALLOWED_DIFFICULTY:
            errors.append(f"المثال {i}: difficulty يجب أن تكون easy/medium/hard")
        if row.get("naturalness") not in range(1, 6):
            errors.append(f"المثال {i}: naturalness يجب أن تكون من 1 إلى 5")
    if errors:
        raise ValueError("فشل التحقق:\n- " + "\n- ".join(errors[:40]))
    return True

validate_triplets(TRIPLETS)
print("نجح التحقق من البيانات ✅")

نجح التحقق من البيانات ✅


## 5) تحميل GATE وإنشاء التضمينات

ينتج النموذج متجهاً من 768 بُعداً لكل جملة. نطبّع المتجهات، ثم يصبح حاصل الضرب بينها هو **Cosine similarity**.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION)

records = []
for i, row in enumerate(TRIPLETS, 1):
    triplet_id = f"{PARTICIPANT_ID}_T{i:02d}"
    for role in ROLES:
        records.append({
            "participant_id": PARTICIPANT_ID,
            "triplet_id": triplet_id,
            "domain": DOMAIN,
            "domain_ar": DOMAIN_AR,
            "role": role,
            "text": " ".join(row[role].split()),
            "hard_negative_mechanism": row["hard_negative_mechanism"],
            "difficulty": row["difficulty"],
            "naturalness": row["naturalness"],
            "notes": row.get("notes", "")
        })

sentences = [r["text"] for r in records]
embeddings = model.encode(
    sentences, batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
print(f"تمثيل {len(sentences)} جملة ← {embeddings.shape[1]} بُعداً ✅")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.44k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

تمثيل 32 جملة ← 768 بُعداً ✅


## 6) هل نجح كل مثال؟

نستخدم معيارين:

- **النجاح الأساسي:** $sim(A,P) > sim(A,HN)$ و $sim(A,P) > sim(A,EN)$.
- **الترتيب الكامل (أصعب):** $sim(A,P) > sim(A,HN) > sim(A,EN)$.

السالب الصعب قد يكون أحياناً أبعد من السالب السهل؛ لذلك لا نستخدم الترتيب الكامل وحده للحكم.

In [ ]:
rows = []
for i, row in enumerate(TRIPLETS):
    start = i * 4
    A, P, HN, EN = embeddings[start:start+4]
    ap, ah, ae = float(A @ P), float(A @ HN), float(A @ EN)
    rows.append({
        "triplet_id": f"{PARTICIPANT_ID}_T{i+1:02d}",
        "sim_anchor_positive": ap,
        "sim_anchor_hard_negative": ah,
        "sim_anchor_easy_negative": ae,
        "positive_margin_over_hard": ap - ah,
        "positive_margin_over_easy": ap - ae,
        "triplet_success": ap > ah and ap > ae,
        "full_order_success": ap > ah > ae,
        "hard_negative_mechanism": row["hard_negative_mechanism"],
        "difficulty": row["difficulty"]
    })

score_df = pd.DataFrame(rows)
display(score_df.style.format({c: "{:.3f}" for c in score_df.columns if c.startswith(("sim_", "positive_"))}))
print(f"النجاح الأساسي: {score_df.triplet_success.mean():.1%}")
print(f"الترتيب الكامل: {score_df.full_order_success.mean():.1%}")

,triplet_id,sim_anchor_positive,sim_anchor_hard_negative,sim_anchor_easy_negative,positive_margin_over_hard,positive_margin_over_easy,triplet_success,full_order_success,hard_negative_mechanism,difficulty
0,P110_T01,0.650,0.335,0.382,0.315,0.268,True,False,changed_location,easy
1,P110_T02,0.764,0.476,0.488,0.287,0.276,True,False,changed_time,easy
2,P110_T03,0.556,0.309,0.594,0.246,-0.038,False,False,changed_condition,medium
3,P110_T04,0.244,0.377,0.175,-0.133,0.069,False,False,changed_entity,medium
4,P110_T05,0.635,0.604,0.373,0.031,0.262,True,True,changed_time,medium
5,P110_T06,0.647,0.464,0.247,0.183,0.400,True,True,changed_condition,hard
6,P110_T07,0.540,0.517,0.212,0.023,0.327,True,True,changed_intent,hard
7,P110_T08,0.331,0.372,0.377,-0.041,-0.046,False,False,negation,hard


النجاح الأساسي: 62.5%
الترتيب الكامل: 37.5%


## 7) مقارنة درجات التشابه

In [ ]:
long_scores = score_df.melt(
    id_vars="triplet_id",
    value_vars=["sim_anchor_positive", "sim_anchor_hard_negative", "sim_anchor_easy_negative"],
    var_name="relation", value_name="cosine_similarity"
)
label_map = {
    "sim_anchor_positive": "Anchor–Positive",
    "sim_anchor_hard_negative": "Anchor–Hard negative",
    "sim_anchor_easy_negative": "Anchor–Easy negative"
}
long_scores["relation"] = long_scores["relation"].map(label_map)
fig = px.bar(long_scores, x="triplet_id", y="cosine_similarity", color="relation", barmode="group",
             range_y=[-0.1, 1], title="Cosine similarity لكل مثال")
fig.update_layout(xaxis_title="المثال", yaxis_title="Cosine similarity", legend_title="العلاقة")
fig.show()

## 8) خريطة الفضاء الدلالي ثنائية الأبعاد

UMAP/PCA أدوات عرض فقط: القرب في الخريطة قد يتشوّه عند ضغط 768 بُعداً إلى بُعدين. اعتمد درجات cosine الأصلية للحكم الدقيق.

In [ ]:
try:
    import umap
    reducer = umap.UMAP(
        n_components=2, n_neighbors=min(10, len(sentences)-1),
        min_dist=0.15, metric="cosine", random_state=42
    )
    xy = reducer.fit_transform(embeddings)
    projection_name = "UMAP"
except Exception as exc:
    from sklearn.decomposition import PCA
    xy = PCA(n_components=2, random_state=42).fit_transform(embeddings)
    projection_name = "PCA"
    print("تعذر UMAP؛ تم استخدام PCA:", exc)

viz_df = pd.DataFrame(records)
viz_df["x"], viz_df["y"] = xy[:, 0], xy[:, 1]
viz_df["role_ar"] = viz_df.role.map(ROLE_AR)

fig = px.scatter(
    viz_df, x="x", y="y", color="triplet_id", symbol="role",
    hover_data={"text": True, "role_ar": True, "x": ":.3f", "y": ":.3f"},
    title=f"{projection_name}: الفضاء الدلالي لجمل {DOMAIN_AR}"
)
for triplet_id, group in viz_df.groupby("triplet_id"):
    anchor = group[group.role == "anchor"].iloc[0]
    positive = group[group.role == "positive"].iloc[0]
    fig.add_trace(go.Scatter(
        x=[anchor.x, positive.x], y=[anchor.y, positive.y], mode="lines",
        line=dict(color="rgba(80,80,80,.35)", width=1),
        hoverinfo="skip", showlegend=False
    ))
fig.update_traces(marker=dict(size=12, line=dict(width=1, color="white")))
fig.update_layout(xaxis_title=f"{projection_name}-1", yaxis_title=f"{projection_name}-2")
fig.show()

## 9) مصفوفة التشابه لمثال واحد

غيّر `SELECTED_TRIPLET` من 1 إلى 8.

In [ ]:
SELECTED_TRIPLET = 1
assert 1 <= SELECTED_TRIPLET <= N_TRIPLETS
start = (SELECTED_TRIPLET - 1) * 4
local_embeddings = embeddings[start:start+4]
matrix = cosine_similarity(local_embeddings)
labels = [ROLE_AR[r] for r in ROLES]
fig = px.imshow(matrix, x=labels, y=labels, text_auto=".3f", zmin=-1, zmax=1,
                color_continuous_scale="RdBu_r", title=f"مصفوفة التشابه — المثال {SELECTED_TRIPLET}")
fig.show()

## 10) أقرب الجمل

اختر رقم أي جملة من الجدول، ثم راقب هل يعيد النموذج الجملة الموجبة الصحيحة أم جملة أخرى.

In [ ]:
sentence_table = viz_df[["triplet_id", "role", "text"]].copy()
sentence_table.index.name = "sentence_index"
display(sentence_table)

QUERY_INDEX = 0
TOP_K = 5
all_sim = embeddings @ embeddings[QUERY_INDEX]
neighbors = np.argsort(-all_sim)
neighbors = [i for i in neighbors if i != QUERY_INDEX][:TOP_K]
neighbor_df = sentence_table.iloc[neighbors].copy()
neighbor_df.insert(0, "cosine_similarity", all_sim[neighbors])
print("الاستعلام:", sentence_table.iloc[QUERY_INDEX].text)
display(neighbor_df)

,triplet_id,role,text
sentence_index,,,
0,P110_T01,anchor,ما هي عاصمة الدولة العباسية في أوج عصرها الذهبي؟
1,P110_T01,positive,بغداد هي التي كانت عاصمة العباسيين ومقر حكمهم في العصر الذهبي.
2,P110_T01,hard_negative,سامراء هي المدينة التي اتخذها بعض خلفاء بني العباس عاصمة لفترة وجيزة.
3,P110_T01,easy_negative,القاهرة كانت عاصمة الدولة الفاطمية في مصر.
4,P110_T02,anchor,ما هي المعركة التاريخية التي أنهت الوجود الإسلامي في الأندلس عام 1492م؟
5,P110_T02,positive,سقوط غرناطة وتسليمها للملوك الكاثوليك هو الحدث الذي أنهى الوجود الإسلامي في الأندلس عام 1492م.
6,P110_T02,hard_negative,معركة بلاط الشهداء كانت معركة حاسمة توقف بعدها التمدد الإسلامي في عمق القارة الأوروبية.
7,P110_T02,easy_negative,معركة حطين كانت معركة فاصلة بين المسلمين والكرادلة/الصليبيين في الشرق الأدنى.
8,P110_T03,anchor,ما هي الأسباب المباشرة التي أدت إلى اندلاع الحرب العالمية الأولى عام 1914؟


الاستعلام: ما هي عاصمة الدولة العباسية في أوج عصرها الذهبي؟


,cosine_similarity,triplet_id,role,text
sentence_index,,,,
1,0.650154,P110_T01,positive,بغداد هي التي كانت عاصمة العباسيين ومقر حكمهم في العصر الذهبي.
3,0.381869,P110_T01,easy_negative,القاهرة كانت عاصمة الدولة الفاطمية في مصر.
2,0.335050,P110_T01,hard_negative,سامراء هي المدينة التي اتخذها بعض خلفاء بني العباس عاصمة لفترة وجيزة.
4,0.283326,P110_T02,anchor,ما هي المعركة التاريخية التي أنهت الوجود الإسلامي في الأندلس عام 1492م؟
20,0.276791,P110_T06,anchor,كيف ساهمت أزمة الكساد الكبير عام 1929 في صعود التيار النازي إلى السلطة في ألمانيا؟
